# Hungarian Dataset Simulation and Optimization

In [ ]:
from utils import data_path, tech_colors, cost_params, week_numbers, seasons
from utils import plot_generator_t, plot_generator_t_plotly, change_costs

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pypsa

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Create analyzed dates
- **week 15**: April 8 - April 14
- **week 28**: July 08 - July 14
- **week 40**: September 30 - October 06
- **week 49**: December 02 - December 08

In [ ]:
dates = []
start_date = pd.to_datetime('2000-01-01') # dummy start date
for week in week_numbers.values():
    s = start_date + pd.Timedelta(weeks=week-1)
    e = s + pd.Timedelta(days=6, hours=23)
    dates.append(pd.date_range(start=s ,end=e, freq='h').to_list())

# Import Data

In [ ]:
potentials_generator = pd.read_excel(data_path, sheet_name='potentials_generator', index_col=0, na_values='None')
potentials_storage = pd.read_excel(data_path, sheet_name='potentials_storage', index_col=0, na_values='None')
costs_generator = pd.read_excel(data_path, sheet_name='costs_generator', index_col=0, na_values='None').to_dict()
costs_storage = pd.read_excel(data_path, sheet_name='costs_storage', index_col=0, na_values='None').to_dict()
profile_wind = pd.read_excel(data_path, sheet_name='profile_wind', names=list(week_numbers.keys()), header=None, index_col=0)
profile_PV = pd.read_excel(data_path, sheet_name='profile_PV', names=list(week_numbers.keys()), header=None, index_col=0)
demand = pd.read_excel(data_path, sheet_name='demand').values.ravel()

#### Changing cost paramteres to time dependent cost
- In the future it will be replaced with stock market data.

In [ ]:
change_costs(costs_generator, ['Nuclear', 'Solar'], ['operation'], 
             function=np.random.uniform, low=0.95, high=1.05, size=demand.shape)

# PyPSA - Network & Optimizations

- [Generator parameters](https://docs.pypsa.org/latest/api/components/types/generators/#pypsa.components.Generators.add) 
- [Storage Unit paramaters](https://docs.pypsa.org/latest/api/components/types/storage-units/#pypsa.components.StorageUnits.add)
    <!--     - `p_nom` / `p_nom_extendable`: nominális teljesítmény (korlátoztható/bővíthető)
    - `p_nom_min` / `p_nom_max` / `p_nom_set`
    - `p_min_pu` / `p_max_pu`: min input / max output for each snapshot
    --->




- [HiGHS](https://en.wikipedia.org/wiki/HiGHS_optimization_solver)

In [ ]:
# Building the network
network = pypsa.Network(name='Network')

# Time stamps
snapshots = np.array([item for sublist in dates for item in sublist])
network.set_snapshots(list(snapshots))

# Base electrical network
network.add(class_name="Bus", 
            name="country_0",
            carrier='AC'
           )

# Set a carrier for the network
network.add(class_name='Carrier',
            name='AC'
           )

# Add demand to the network
network.add(class_name="Load", 
            name="Residential demand",  
            bus="country_0", 
            p_set=demand
           )   

# Add generators to the network
for technology in potentials_generator.keys(): 
    p_max_pu=1
    committable=False

    # getting profiles 
    if technology in ['Solar']:
        p_max_pu=np.repeat(profile_PV.values, 7, axis=1).flatten('F') # column-major order
    elif technology in ['Wind Onshore']:
        p_max_pu=np.repeat(profile_wind.values, 7, axis=1).flatten('F')
    
    # adding generators
    network.add(class_name="Generator", 
                name=str(technology),
                bus="country_0",
                
                p_nom_extendable=True, # optimisable generator capacity
                p_nom_max=potentials_generator.loc['p_nom_max'][str(technology)], # maximum limit
                p_nom_min=potentials_generator.loc['p_nom_min'][str(technology)], # minimum limit
                capital_cost=costs_generator[str(technology)]['capital'], # generator installation cost
                marginal_cost=sum([costs_generator[str(technology)][key] for key in cost_params]), # operating cost
                p_max_pu=p_max_pu, # maximum power per-unit // production profiles
                ramp_limit_up=potentials_generator.loc['ramp_up'][str(technology)],   # maximum increase per hour
                ramp_limit_down=potentials_generator.loc['ramp_down'][str(technology)], # maximum decrease per hour

                committable=committable,
                ramp_limit_start_up=0.75,
                ramp_limit_shut_down=0.75
            )
   
# Storage units creation
for technology in potentials_storage.keys():
    network.add(class_name="StorageUnit", 
                name=str(technology),
                bus="country_0",
                
                p_nom_extendable=True, # optimisable storage capacity
                p_nom_max=potentials_storage.loc['p_nom_max'][str(technology)], # maximum limit
                capital_cost=costs_storage[str(technology)]['capital'], # installation cost
                marginal_cost=sum([costs_storage[str(technology)][key] for key in cost_params]), # operating cost
            )
    
#network.generators.loc['Waste', 'committable'] = True
#network.generators_t.shut_down['Waste'] = (snapshots == pd.Timestamp('2000-04-08 18:00:00')).astype(int)
#network.generators_t.start_up['Waste'] = (snapshots == pd.Timestamp('2000-04-10 18:00:00')).astype(int)

#maintenance = (~(snapshots>pd.Timestamp('2000-04-08 15:00:00'))|~(snapshots<pd.Timestamp('2000-04-11 10:00:00'))).astype(int)
#network.generators_t.status['Waste'] = maintenance

In [ ]:
# Optimization
network.sanitize()
network.optimize(solver_name='highs', 
                 solver_options={'presolve': 'on', 
                                 'threads': 8,
                                 'time_limit': 3600,
                                 'method': 'ipm',
                                 'run_crossover': 'off'}
                )

#_ = network.export_to_netcdf("data/power_network_HU.nc")

In [ ]:
plot_generator_t(network, dates, season='Spring', day=1, colors=tech_colors)

In [ ]:
# create interactive plots for each season
for season in seasons.keys():
    plot_generator_t_plotly(network,  
                            dates,
                            season,
                            colors=tech_colors
                            )